In [24]:
!python --version

Python 3.10.4


In [25]:
%pip install matplotlib nest-asyncio openai pandas python-dotenv safetensors scikit-learn torch tiktoken tqdm 

Note: you may need to restart the kernel to use updated packages.


In [37]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [27]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [28]:
import sqlite3
conn = sqlite3.connect(f'../train_embeddings.db')

In [29]:
cursor = conn.cursor()
cursor.execute("""
  CREATE TABLE IF NOT EXISTS comments_gpt4_annotated (
    id TEXT PRIMARY KEY,
    comment_text TEXT,
    toxic INTEGER,
    severe_toxic INTEGER,
    obscene INTEGER,
    threat INTEGER,
    insult INTEGER,
    identity_hate INTEGER
);
""")
conn.commit()

In [30]:
cursor = conn.cursor()
cursor.execute("SELECT * FROM comments")
comments = cursor.fetchall()

In [38]:
import openai
import os

openai.api_key = os.environ["OPENAI_API_KEY"]  

def annotate_comment(comment):
    response = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {
            "role": "system",
            "content": "You are a content moderator an score user generated comments based on the following criteria:\n\nToxic: very bad, unpleasant, or harmful\nSevere toxic: extremely bad and offensive\nObscene: (of the portrayal or description of sexual matters) offensive or disgusting by accepted standards of morality and decency\nThreat: a statement of an intention to inflict pain, injury, damage, or other hostile action on someone in retribution for something done or not done\nInsult: speak to or treat with disrespect or scornful abuse\nIdentity hate: hatred, hostility, or violence towards members of a race, ethnicity, nation, religion, gender, gender identity, sexual orientation or any other designated sector of society\n\nQ: Hello World\nA: Toxic: 0, Severe Toxic: 0, Obscene: 0, Threat: 0, Insult: 0, Identity Hate: 0\n\nQ: COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK\nA: Toxic: 1, Severe Toxic: 1, Obscene: 1, Threat: 0, Insult: 1, Identity Hate: 0"
            },
            {
            "role": "user",
            "content": f"Q: {comment}\nA:"
            },
        ],
        temperature=1,
        max_tokens=256,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0
    )
    return response

{
  "id": "chatcmpl-7oLVU7MdJabd7fL2KyU4i5klVxRvE",
  "object": "chat.completion",
  "created": 1692233428,
  "model": "gpt-4-0613",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "Toxic: 0, Severe Toxic: 0, Obscene: 0, Threat: 0, Insult: 0, Identity Hate: 0"
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 315,
    "completion_tokens": 35,
    "total_tokens": 350
  }
}

In [39]:
import asyncio
import nest_asyncio
from tqdm import trange

nest_asyncio.apply()
batch_size = 3

for i in trange(0, len(comments), batch_size):
  batch = comments[i:i+batch_size]
  batch_ids = [x[0] for x in batch]
  batch_text = [x[1] for x in batch]

  cursor.execute(
    """
          SELECT id FROM comments_gpt4_annotated WHERE id IN (%s)
    """ % ','.join('?'*len(batch_ids)), batch_ids
  )
  annotated_ids = [x[0] for x in cursor.fetchall()]
  tasks = []
  loop = asyncio.get_event_loop()
  for item in batch:
    id = item[0]
    text = item[1]
    if id not in annotated_ids:
      task = loop.run_in_executor(None, annotate_comment, text)
      tasks.append(task)
    else:
      print("Already annotated comment with id: ", id)
  
  for response in await asyncio.gather(*tasks):
    # Update the database
    annotation = response.choices[0].message.content
    annotation = annotation.split(", ")
    values = [x.split(": ")[1] for x in annotation]

    # Update the database
    cursor.execute(
      """
            INSERT OR REPLACE INTO comments_gpt4_annotated (id, comment_text, toxic, severe_toxic, obscene, threat, insult, identity_hate)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
      """,
      (id, text, values[0], values[1], values[2], values[3], values[4], values[5])
    )
  conn.commit()

  0%|          | 0/53191 [00:00<?, ?it/s]

Already annotated comment with id:  0000997932d777bf
Already annotated comment with id:  000103f0d9cfb60f
Already annotated comment with id:  000113f07ec002fd
Already annotated comment with id:  0001b41b1c6bb37e
Already annotated comment with id:  0001d958c54c6e35
Already annotated comment with id:  00025465d4725e87
Already annotated comment with id:  0002bcb3da6cb337
Already annotated comment with id:  00031b1e95af7921
Already annotated comment with id:  00037261f536c51d
Already annotated comment with id:  00040093b2687caa
Already annotated comment with id:  0005300084f90edc
Already annotated comment with id:  00054a5e18b50dd4
Already annotated comment with id:  0005c987bdfc9d4b
Already annotated comment with id:  0006f16e4e9f292e
Already annotated comment with id:  00070ef96486d6f9
Already annotated comment with id:  00078f8ce7eb276d
Already annotated comment with id:  0007e25b2121310b
Already annotated comment with id:  000897889268bc93
Already annotated comment with id:  0009801bd8

  0%|          | 18/53191 [00:04<3:38:38,  4.05it/s]

Already annotated comment with id:  0020fd96ed3b8c8b


  0%|          | 19/53191 [00:08<7:14:38,  2.04it/s]

...

Already annotated comment with id:  002a6beca33307b3


  0%|          | 24/53191 [00:25<31:20:32,  2.12s/it]

Already annotated comment with id:  002d6c9d9f85e81f


  0%|          | 28/53191 [00:38<20:17:11,  1.37s/it]


RateLimitError: Rate limit reached for 10KTPM-200RPM in organization org-ysWxOFiDLrX8qwoZVxN3vgOV on tokens per min. Limit: 10000 / min. Please try again in 6ms. Contact us through our help center at help.openai.com if you continue to have issues.